# 03. Statistics & ML Pipeline

- 기술통계(평균·표준편차·분위수), 상관계수 산출
- `scipy.stats.ttest_ind` 로 t-test 수행 및 p-value 해석
- `sklearn.pipeline.Pipeline` 으로 전처리+모델 학습, 평가 지표 출력, `joblib` 모델 저장

In [ ]:
import sys

sys.path.append("..")

from pathlib import Path

import pandas as pd

from src import ml, stats

In [ ]:
card = pd.read_parquet(Path("..") / "data" / "processed" / "card_clean.parquet")
card.shape

## 1. 기술통계 · 상관계수

In [ ]:
stats.describe_stats(card, ["trip_distance", "fare_amount", "tip_amount", "tip_pct"])

In [ ]:
stats.correlation_matrix(card, ["trip_distance", "fare_amount", "trip_duration_min", "tip_pct"])

## 2. t-test — 시간대·거리·혼잡구역·승객수·요금이 팁에 미치는 영향

표본이 265만 건 이상이라 p-value는 거의 항상 유의하게 나온다. 그래서 p<0.05 여부와 별개로
**효과크기(Cohen's d / r)** 로 실질적 중요도를 함께 판단한다
(|d| < 0.1 무시가능, < 0.3 작음, < 0.5 중간, 그 이상 큼).

In [ ]:
sig_df = stats.run_significance_suite(card)
sig_df

## 3. ML Pipeline

두 개의 타겟을 각각 예측한다:
- `has_tip` — 팁을 주는지 여부 (주제의 "지급률"에 대응)
- `high_tip` — 팁 비율 20% 이상 (주제의 "팁 비율"에 대응)

`sklearn.pipeline.Pipeline`으로 전처리(StandardScaler + OneHotEncoder) + 모델(LogisticRegression,
RandomForest)을 구성하고, `joblib`으로 `models/`에 저장한다.

In [ ]:
models_dir = Path("..") / "models"

ml_results = {}
for target_col, cfg in ml.TARGET_CONFIGS.items():
    print(f"--- {cfg['label']} ---")
    num_features = cfg["num_features"]
    ml_results[target_col] = ml.run_target_pipeline(card, target_col, num_features, models_dir)

In [ ]:
summary = pd.DataFrame(
    [
        {
            "target": target_col,
            "label": ml.TARGET_CONFIGS[target_col]["label"],
            "baseline_acc": res["baseline_acc"],
            "best_accuracy": res["metrics_df"]["accuracy"].max(),
            "macro_f1": res["metrics_df"]["macro_f1"].max(),
            "mcfadden_r2": res["mcfadden"]["r2"],
        }
        for target_col, res in ml_results.items()
    ]
)
summary

**해석**: `has_tip`(McFadden R² ≈ 0.44)이 `high_tip`(McFadden R² ≈ 0.13)보다 훨씬 설명력이 높다.
즉 누가 팁을 주는지는 잘 설명되지만, 얼마나 후하게 주는지는 잘 설명되지 않는다 — 통계 검정에서
혼잡구역·VendorID 외 대부분의 변수 효과크기가 tip_pct에 대해서는 작았던 것과 일치한다.

## 4. report.md 생성

In [ ]:
from src import load, report

card_binned = card.copy()
card_binned["distance_bin"] = pd.cut(card_binned["trip_distance"], bins=[0, 1, 3, 5, 10, 20, 1000])

eda_tables = {
    "시간대별": stats.groupby_tip_summary(card_binned, "pickup_hour"),
    "거리구간별": stats.groupby_tip_summary(card_binned, "distance_bin"),
    "혼잡구역여부별": stats.groupby_tip_summary(card_binned, "is_congestion"),
    "승객수별": stats.groupby_tip_summary(card_binned, "passenger_count"),
    "주중_주말별": stats.groupby_tip_summary(card_binned, "is_weekend"),
}

for target_col, res in ml_results.items():
    res["label"] = ml.TARGET_CONFIGS[target_col]["label"]

report_path = Path("..") / "reports" / "report.md"
report.generate_report(load.compare_loaders(), eda_tables, sig_df, ml_results, report_path)
print(f"report.md 생성 완료: {report_path.resolve()}")